In [7]:
from utils.dynamicRieszFunctions import estimateDynamicRiesz_all
from utils.dynamicRieszFunctions import estimateDynamicRiesz
import torch
import pandas as pd
import time
from torch.distributions import Normal
from utils.dgp import DiD_DGP
from tqdm import tqdm

## Settings


In [8]:
lasso_cv_settings = {
    'b_degree' : 1,
    'cv_folds' : 5,
    'random_state' : 42
}

lasso_a_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' : "CV",
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

lasso_f_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' :  "CV",
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

rf_a_settings = {
    'poly_degree' : 0,
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : -1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}
rf_f_settings = {
    'poly_degree' : 1, # 1 or 2?
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : -1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}

net_a_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}

net_f_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}


## Coefficients and Parameters

In [9]:
# Parameters
N = 1000
tmax = 10
dimX = 3
dimZ = 2
seed = 0
folds = 4

## Propensity models 

In [10]:
# Bounds (only for truncated distributions)
lower = 0.30
upper = 0.90

# Define logistic function
def logistic(x):
    return torch.exp(x) / (1 + torch.exp(x))

# Define a truncated logistic function
def truncated_logistic(x):
    return lower + (upper - lower) * logistic(x)

# Simple nonlinear probability (from adversarial Riesz paper)
def truncated_adv(x):
    return lower + (upper - lower) * (x > 0)


## Generate data

In [5]:
dgp = DiD_DGP(dim_X=dimX, dim_Z=dimZ, 
                alpha_1 = .5, # Y_1s effect on D
              gamma_1=torch.ones(dimZ), # Z effect on D
                gamma_2=torch.ones(dimZ), # Z effect on Y
                g = truncated_adv # specification of the propensity modek
                )

data = dgp.generate(n =N*tmax, seed = seed)
X1 = data['X1']
X2 = data['X2']
Y1 = data['Y1']
Y2 = data['Y2']
Z = data['Z']
D = data['D']
ATT = data['ATT']

In [6]:
print(X1.shape)
print(X2.shape)
print(Y1.shape)
print(Y2.shape)
print(Z.shape)
print(D.shape)

torch.Size([10000, 3])
torch.Size([10000, 3])
torch.Size([10000])
torch.Size([10000])
torch.Size([10000, 2])
torch.Size([10000])


In [7]:
print(ATT)
print(D.mean())

tensor(2.8500)
tensor(0.6335)


## Estimation

In [8]:
time0 = time.time()
pred_theta = torch.zeros(tmax,18)
pred_sig = torch.zeros(tmax,18)

RR1 = torch.zeros(N,tmax,18)
RR2 = torch.zeros(N,tmax,18)
f1 = torch.zeros(N,tmax,18)
f2 = torch.zeros(N,tmax,18)

for t in tqdm(range(0,tmax)):

    # Get data for current iteration
    X1_sub = X1[t*N:(t+1)*N,:]
    X2_sub = X2[t*N:(t+1)*N,:]
    Y1_sub = Y1[t*N:(t+1)*N].view(-1, 1)
    Y2_sub = Y2[t*N:(t+1)*N].view(-1, 1)
    D_sub = D[t*N:(t+1)*N].view(-1, 1)
    Z_sub = Z[t*N:(t+1)*N,:]




    pred_theta[t,3], pred_sig[t,3], RR1[:,t,3:4], RR2[:,t,3:4] = estimateDynamicRiesz(Y1_sub, Y2_sub, D_sub, Z_sub, X1_sub  , X2_sub, folds,
                                                                     method_a = "RF", rf_a_settings=rf_a_settings,
                                                                        method_f = "RF", rf_f_settings = rf_f_settings)





100%|██████████| 10/10 [00:21<00:00,  2.15s/it]


In [9]:
pred_sig[:,3] 

tensor([3.3389, 3.3613, 3.1634, 3.1602, 3.1687, 3.3014, 3.1933, 3.2804, 3.0929,
        3.3122])

In [10]:
pred_sig[:,3] - ATT # bias

tensor([0.4889, 0.5113, 0.3134, 0.3102, 0.3187, 0.4514, 0.3433, 0.4304, 0.2429,
        0.4622])

In [11]:
torch.mean(pred_sig[:,3] - ATT) # bias

tensor(0.3873)

## Application

In [1]:
from utils.application import *#
data_apllication = application_data()

In [2]:
data_dict = data_apllication.get_data(2003, 2004 )

Changing covariates:  ['emp0A01_BS', 'pop', 'annual_avg_pay', 'lpop', 'lavg_pay']
Z variables:  ['region_2', 'region_3', 'region_4', 'censusdiv_3', 'censusdiv_4', 'censusdiv_5', 'censusdiv_6', 'censusdiv_7', 'censusdiv_8']


In [3]:
X1 = data_dict['X1']
X2 = data_dict['X2']
Y1 = data_dict['Y1']
Y2 = data_dict['Y2']
D = data_dict['D']
Z = data_dict['Z']  

In [4]:
D.sum()

tensor(102.)

In [5]:
X1.shape

torch.Size([2491, 5])

In [11]:
estimateDynamicRiesz(Y1, Y2, D, Z, X1, X2, folds,
                                                                     method_a = "RF", rf_a_settings=rf_a_settings,
                                                                        method_f = "RF", rf_f_settings = rf_f_settings)

(tensor(-0.0220),
 tensor(1.0146),
 tensor([[1.0629],
         [1.0384],
         [1.0585],
         ...,
         [1.0571],
         [1.0778],
         [1.0720]]),
 tensor([[1.0497],
         [1.0377],
         [1.0507],
         ...,
         [1.0482],
         [1.0575],
         [1.0584]]))

In [3]:
test =pd.read_csv("https://raw.githubusercontent.com/CausalAIBook/MetricsMLNotebooks/main/data/minwage_data.csv", index_col=0)

In [13]:
Z = test[test["year"] == 2003][["region", "censusdiv"]].reset_index(drop=True)

test["region"].value_counts()

region
3    7469
2    7301
4    2730
1    1351
Name: count, dtype: int64

In [14]:
Z["region"].value_counts()

region
3    1067
2    1043
4     390
1     193
Name: count, dtype: int64

In [15]:
Z["censusdiv"].value_counts()

censusdiv
4    606
5    584
3    437
7    388
8    234
9    156
2    150
6     95
1     43
Name: count, dtype: int64